In [2]:
""" Definitions used throughout the package"""
#import modules
from desidlas.datasets.preprocess import estimate_s2n,normalize,rebin
from desidlas.datasets.DesiMock import DesiMock
#from desidlas.dla_cnn.defs import best_v
import numpy as np
import os
from os.path import join
from pkg_resources import resource_filename
from pathlib import Path
#from desidlas.datasets.get_sightlines import get_sightlines
#REST_RANGE = [900, 1346, 1748]
#kernel = 400 # SDSS value -- UPDATE!!
#smooth_kernel = 600
#best_v = {'b': 62996, 'r': 44859, 'z': 34720, 'all': 44735}#the best value of rebining for each channel, its unit is m*s^(-1).

# Stacking after all runnings

In [4]:
sightline_total = '/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark'

list1 = os.listdir(sightline_total)
sightline_list=[]
partpre_list=[]
dlacat_list=[]

list1 = os.listdir(sightline_total)
sightline_list=[]
partpre_list=[]
dlacat_list=[]
for k in list1:
    list2=os.listdir(sightline_total+'/'+str(k)) 
    for j in list2:
        if j.endswith('-dlacat.fits'):
            dlacat_list.append(sightline_total+'/'+str(k)+'/'+j) 
dlacat_total_list = np.array(dlacat_list)

print(dlacat_total_list.shape)  

(16286,)


In [5]:
dlacat_total_list

array(['/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark/705/jura-main-dark-7-705-dlacat.fits',
       '/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark/21826/jura-main-dark-218-21826-dlacat.fits',
       '/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark/5937/jura-main-dark-59-5937-dlacat.fits',
       ...,
       '/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark/28574/jura-main-dark-285-28574-dlacat.fits',
       '/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark/10144/jura-main-dark-101-10144-dlacat.fits',
       '/global/cfs/cdirs/desi/users/jqzou/jura/sightlines_main_dark/18558/jura-main-dark-185-18558-dlacat.fits'],
      dtype='<U103')

In [6]:
from astropy.table import vstack, Table
base_table = Table.read(dlacat_total_list[0], format='fits')
j=0
for dlacat in dlacat_total_list[1:]:
    try:
        append_table = Table.read(dlacat, format='fits')
        # Read in the large table you want to append to 
        # Use Astropy's 'vstack' function and overwrite the file 
        base_table = vstack([base_table,append_table])
        j=j+1
    except:
        print(dlacat)
        continue
base_table.write('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/data/Jura/dlacat.fits', format='fits', overwrite=True)

In [4]:
j

1028

In [11]:
from fitsio import FITS
f=Table.read('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/data/Jura/dlacat.fits')

In [12]:
f

TARGET_RA,TARGET_DEC,Z_QSO,Z_DLA,TARGETID,S2N,DLAID,NHI,DLA_CONFIDENCE,NHI_STD,ABSORBER_TYPE
float64,float64,float64,float64,int64,float64,bytes21,float64,float64,float64,bytes6
34.38134011338037,20.19389241203372,3.0480521328473693,2.2352922951539282,39628265064761297,14.411231994628906,39628265064761297000,19.563732147216797,0.225,0.1968216896057129,SUBDLA
34.38134011338037,20.19389241203372,3.0480521328473693,2.3367774041849194,39628265064761297,14.411231994628906,39628265064761297001,19.8466739654541,0.4125,0.12255758047103882,SUBDLA
34.38134011338037,20.19389241203372,3.0480521328473693,2.8841338617382726,39628265064761297,14.411231994628906,39628265064761297002,19.537622451782227,0.375,0.11430080235004425,SUBDLA
34.38134011338037,20.19389241203372,3.0480521328473693,2.9958234577425698,39628265064761297,14.411231994628906,39628265064761297003,19.830015182495117,0.675,0.12394750118255615,SUBDLA
34.3991282995279,20.29295496953802,2.667444086673286,2.515664519413663,39628265064761691,0.4227898120880127,39628265064761691000,20.55319595336914,0.4875,0.2509342133998871,DLA
33.94298327637199,20.61511241807015,2.22327608776504,2.5330179502327366,39628270735459757,0.6772162914276123,39628270735459757000,19.829288482666016,0.2375,0.08915319293737411,SUBDLA
34.1609433525394,20.465926130448185,2.519015291104234,2.1646307453947102,39628270739653364,2.5772275924682617,39628270739653364000,19.829540252685547,0.5625,0.08845075219869614,SUBDLA
34.225438161412576,20.59549027819075,2.764163578715544,2.128480131965388,39628270739654811,1.9532759189605713,39628270739654811000,19.839832305908203,0.275,0.14914502203464508,SUBDLA
34.225438161412576,20.59549027819075,2.764163578715544,2.3174166094616298,39628270739654811,1.9532759189605713,39628270739654811001,19.973012924194336,1.0,0.14455373585224152,SUBDLA


In [18]:
len(f[f['DLA_CONFIDENCE']>0.3]),len(f[(f['DLA_CONFIDENCE']>0.3)&(f['NHI']>20)]),len(f[(f['DLA_CONFIDENCE']>0.3)&(f['NHI']>20)&(f['S2N']>3)])

(762281, 309803, 117726)

## Different DLA cat for P1D
### NHI>20
### C>0.3, NHI>20
### C>0.3for S2N<3, C>0.2for S2N>3, NHI>20
### C>0.5, NHI>20

In [3]:
from astropy.table import Column
from astropy.table import Table
dla_true=Table.read('/global/cfs/cdirs/desicollab/users/naimgk/ohio-p1d/v2.0/iron/main/QSO_cat_iron_main_dark_healpix_v0/v2.0.0/desi-2.15-1/dla_cat.fits')
z_true=Table.read('/global/cfs/cdirs/desicollab/users/naimgk/ohio-p1d/v2.0/iron/main/QSO_cat_iron_main_dark_healpix_v0/v2.0.0/desi-2.15-1/zcat.fits')
dla_finder=Table.read('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat.fits')
conf = dla_finder['DLA_CONFIDENCE']
NHI=dla_finder['NHI']
S2N=dla_finder['S2N']
tbhdu_Z = Column(name='Z', data=dla_finder['Z_DLA'])
dla_finder.add_column(tbhdu_Z)

In [13]:
Table_all=dla_finder[NHI_finder>20]
Table_1=dla_finder[(NHI_finder>20)&(conf>0.3)]
Table_2=dla_finder[(NHI_finder>20)&(conf>0.3)&(S2N<3)|(NHI_finder>20)&(conf>0.2)&(S2N>3)]
Table_3=dla_finder[(NHI_finder>20)&(conf>0.5)]

In [14]:
Table_all.write('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat_NHI_20.fits', format='fits', overwrite=True)
Table_1.write('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat_NHI_20_cf_0.3.fits', format='fits', overwrite=True)
Table_2.write('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat_NHI_20_cf_0.3_S2N.fits', format='fits', overwrite=True)
Table_3.write('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat_NHI_20_cf_0.5.fits', format='fits', overwrite=True)


In [18]:
from fitsio import FITS

In [19]:
f=FITS('/global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat_NHI_20_cf_0.5.fits')

In [21]:
f[1]


  file: /global/cfs/cdirs/desi/users/tingtan/DLA_finder/mocks/iron_ihio/dlacat_NHI_20_cf_0.5.fits
  extension: 1
  type: BINARY_TBL
  extname: DLACAT
  rows: 76588
  column info:
    TARGET_RA           f8  
    TARGET_DEC          f8  
    Z_QSO               f8  
    Z_DLA               f8  
    TARGETID            i8  
    S2N                 f8  
    DLAID              S20  
    NHI                 f8  
    DLA_CONFIDENCE      f8  
    NHI_STD             f8  
    ABSORBER_TYPE       S6  
    Z                   f8  